# माइक्रोसफ्ट एजेन्ट फ्रेमवर्कसहित मानव-इन-द-लूप कार्यप्रवाह

## 🎯 सिकाइका उद्देश्यहरू

यस नोटबुकमा, तपाईंले Microsoft Agent Framework को `RequestInfoExecutor` प्रयोग गरेर **मानव-इन-द-लूप** कार्यप्रवाह कसरी कार्यान्वयन गर्ने भनेर सिक्नुहुनेछ। यो शक्तिशाली ढाँचा एआई कार्यप्रवाहहरूलाई रोक्न र मानव इनपुट संकलन गर्न अनुमति दिन्छ, जसले तपाइँका एजेन्टहरूलाई अन्तरक्रियात्मक बनाउँछ र मानवलाई महत्वपूर्ण निर्णयहरूमा नियन्त्रण प्रदान गर्दछ।

## 🔄 मानव-इन-द-लूप के हो?

**मानव-इन-द-लूप (HITL)** एक डिजाइन ढाँचा हो जहाँ एआई एजेन्टहरूले कार्यान्वयन रोक्छन् र निरन्तरता अघि मानव इनपुटको अनुरोध गर्छन्। यो आवश्यक छ:

- ✅ **महत्वपूर्ण निर्णयहरू** - महत्वपूर्ण कदम चाल्नु अघि मानव अनुमोदन प्राप्त गर्नुहोस्
- ✅ **अस्पष्ट परिस्थितिहरू** - एआई अनिश्चित हुँदा मान्छेलाई स्पष्ट पार्न दिनुहोस्
- ✅ **प्रयोगकर्ता प्राथमिकताहरू** - प्रयोगकर्ताहरूलाई विभिन्न विकल्पहरूबाट छनोट गर्न सोध्नुहोस्
- ✅ **अनुपालन र सुरक्षा** - नियमित अपरेसनहरूका लागि मानव निगरानी सुनिश्चित गर्नुहोस्
- ✅ **अन्तरक्रियात्मक अनुभवहरू** - प्रयोगकर्ता इनपुटमा प्रतिक्रिया दिने संवादात्मक एजेन्टहरू बनाउनुहोस्

## 🏗️ माइक्रोसफ्ट एजेन्ट फ्रेमवर्कमा यसले कसरी काम गर्छ

यो फ्रेमवर्कले HITL का लागि तीन मुख्य कम्पोनेन्टहरू प्रदान गर्दछ:

1. **`RequestInfoExecutor`** - एउटा विशेष कार्यान्वयनकर्ता जसले कार्यप्रवाहलाई रोक्छ र `RequestInfoEvent` प्रसारण गर्छ
2. **`RequestInfoMessage`** - मानवलाई पठाइने टाइप गरिएको अनुरोध पेलोडहरूको आधार कक्षा
3. **`RequestResponse`** - मानव प्रतिक्रिया र मूल अनुरोधलाई `request_id` प्रयोग गरेर सम्बन्धित गर्दछ

**कार्यप्रवाह ढाँचा:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 हाम्रो उदाहरण: प्रयोगकर्ता पुष्टि सहित होटल बुकिङ

हामी सर्तीय कार्यप्रवाहमा मानव पुष्टि थपेर वैकल्पिक गन्तव्यहरू सुझाव दिनु अघि बनाउनेछौं:

1. प्रयोगकर्ताले गन्तव्यको अनुरोध गर्दछ (जस्तै, "पेरिस")
2. `availability_agent` ले कोठाहरू उपलब्ध छन् कि छैन जाँच गर्छ
3. **यदि कोठा छैन भने** → `confirmation_agent` ले "तपाईलाई विकल्पहरू देखाउनु पर्ने हो?" सोध्छ
4. कार्यप्रवाह `RequestInfoExecutor` प्रयोग गरेर **रुकिन्छ**
5. **मानवले** कन्सोल इनपुटमार्फत "हो" वा "होइन" प्रतिक्रिया दिन्छ
6. `decision_manager` ले प्रतिक्रियाको आधारमा मार्ग निर्धारण गर्छ:
   - **हो** → वैकल्पिक गन्तव्यहरू देखाउनुहोस्
   - **होइन** → बुकिङ अनुरोध रद्द गर्नुहोस्
7. अन्तिम परिणाम प्रदर्शन गर्नुहोस्

यसले प्रयोगकर्ताहरूलाई एजेन्टका सुझावहरूमा नियन्त्रण कसरी दिने देखाउँछ!

---

सुरु गरौँ! 🚀


## चरण 1: आवश्यक पुस्तकालयहरू आयात गर्नुहोस्

हामी मानक एजेन्ट फ्रेमवर्क कम्पोनेन्टहरू आयात गर्छौं साथै **मानव-इन-द-लूप विशिष्ट कक्षाहरू**:
- `RequestInfoExecutor` - मानव इनपुटको लागि कार्यप्रवाह रोक्ने कार्यान्वयनकर्ता
- `RequestInfoEvent` - मानव इनपुट माग्दा उत्सर्जित घटना
- `RequestInfoMessage` - प्रकारित अनुरोध प्यालोडहरूको आधार कक्षा
- `RequestResponse` - मानव प्रतिक्रियाहरूलाई अनुरोधहरूसँग सम्बन्धित गर्ने
- `WorkflowOutputEvent` - कार्यप्रवाह उत्पादनहरू पत्ता लगाउने घटना


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## चरण २: संरचित आउटपुटहरूका लागि पाइडान्टिक मोडेलहरू परिभाषित गर्नुहोस्

यी मोडेलहरूले एजेन्टहरूले फर्काउने **स्किमा** परिभाषित गर्छन्। हामी सर्तगत कार्यप्रवाहबाट सबै मोडेलहरू राख्छौं र थप्छौं:

**मानव-इन-द-लूपका लागि नयाँ:**
- `HumanFeedbackRequest` - `RequestInfoMessage` को उपवर्ग जसले मानिसहरूलाई पठाइने अनुरोध प्यालोड परिभाषित गर्छ
  - `prompt` (सोध्ने प्रश्न) र `destination` (अप्राप्य शहरको बारेमा सन्दर्भ) समावेश गर्दछ


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## चरण ३: होटल बुकिङ उपकरण सिर्जना गर्नुहोस्

सर्तीय कार्यप्रवाहबाट उही उपकरण - गन्तव्यमा कोठाहरू उपलब्ध छन् कि छैनन् जाँच गर्दछ।


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## चरण 4: रुटिङको लागि सर्त कार्यहरू परिभाषित गर्नुहोस्

हामीलाई हाम्रो मानव-इन-द-लूप कार्यप्रवाहको लागि **चार सर्त कार्यहरू** आवश्यक छन्:

**सर्तमूलक कार्यप्रवाहबाट:**
1. `has_availability_condition` - होटलहरू उपलब्ध हुँदा मार्गनिर्देशन गर्छ
2. `no_availability_condition` - होटलहरू उपलब्ध नभएको अवस्थामा मार्गनिर्देशन गर्छ

**मानव-इन-द-लूपको लागि नयाँ:**
3. `user_wants_alternatives_condition` - प्रयोगकर्ताले वैकल्पिक विकल्पहरूलाई "हो" भनेमा मार्गनिर्देशन गर्छ
4. `user_declines_alternatives_condition` - प्रयोगकर्ताले वैकल्पिक विकल्पहरूलाई "होइन" भनेमा मार्गनिर्देशन गर्छ


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## चरण ५: निर्णय प्रबन्धक कार्यान्वयनकर्ता सिर्जना गर्नुहोस्

यो **मानव-इन-द-लूप ढाँचाको मूल हो**! `DecisionManager` एक अनुकूलित `Executor` हो जुन:

१. **मानव प्रतिक्रिया प्राप्त गर्दछ** `RequestResponse` वस्तुहरूको माध्यमबाट
२. **प्रयोगकर्ताको निर्णय प्रशोधन गर्दछ** (हो/होइन)
३. **कार्यप्रवाह सन्देशहरू उपयुक्त एजेन्टहरूमा पठाएर मार्गदर्शन गर्दछ**

मुख्य विशेषताहरू:
- कार्यप्रवाह चरणहरूका रूपमा विधिहरू प्रदर्शन गर्न `@handler` सजावट प्रयोग गर्दछ
- `RequestResponse[HumanFeedbackRequest, str]` प्राप्त गर्दछ जसमा मूल अनुरोध र प्रयोगकर्ताको उत्तर दुबै समावेश छन्
- हाम्रो सर्त कार्यहरूलाई सक्रिय पार्ने सरल "हो" वा "होइन" सन्देशहरू उत्पन्न गर्दछ


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## चरण ६: अनुकूल प्रदर्शन कार्यान्वायकता बनाउनुहोस्

सशर्त कार्यप्रवाहबाट एउटै प्रदर्शन कार्यान्वायकता - कार्यप्रवाह आउटपुटको रूपमा अन्तिम परिणामहरू दिन्छ।


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## चरण ७: वातावरण चरहरू लोड गर्नुहोस्

LLM क्लाइन्ट (Microsoft Foundry / Azure OpenAI) कन्फिगर गर्नुहोस्।


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## चरण ८: AI एजेन्टहरू र कार्यान्वयनकर्ताहरू सिर्जना गर्नुहोस्

हामी **छवटा कार्यप्रवाह कम्पोनेन्टहरू** सिर्जना गर्छौं:

**एजेन्टहरू (AgentExecutor मा समेटिएको):**
1. **availability_agent** - उपकरण प्रयोग गरी होटल उपलब्धता जाँच्छ
2. **confirmation_agent** - 🆕 मानव पुष्टिकरण अनुरोध तयार गर्दछ
3. **alternative_agent** - वैकल्पिक शहरहरू सुझाव दिन्छ (जब प्रयोगकर्ताले हो भन्छ)
4. **booking_agent** - बुकिङ प्रोत्साहित गर्छ (जब कोठाहरू उपलब्ध हुन्छन्)
5. **cancellation_agent** - 🆕 रद्द गर्ने सन्देश व्यवस्थापन गर्दछ (जब प्रयोगकर्ताले होईन भन्छ)

**विशेष कार्यान्वयनकर्ता:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` जसले मानव इनपुटको लागि कार्यप्रवाह रोकिन्छ
7. **decision_manager** - 🆕 मानव प्रतिक्रियाको आधारमा मार्ग निर्देशन गर्ने अनुकूल कार्यान्वयनकर्ता (पहिले नै माथि परिभाषित)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## चरण 9: मानव-इन-द-लूप सहित Workflow निर्माण गर्नुहोस्

अब हामी **शर्तीय मार्ग निर्देशन** सहित मानव-इन-द-लूप पथको साथ workflow ग्राफ निर्माण गर्छौं:

**Workflow संरचना:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**मुख्य एजहरू:**
- `availability_agent → confirmation_agent` (कोठा नभएको अवस्थामा)
- `confirmation_agent → prepare_human_request` (प्रकार रूपान्तरण)
- `prepare_human_request → request_info_executor` (मानवको लागि रोक)
- `request_info_executor → decision_manager` (सधैं - RequestResponse प्रदान गर्दछ)
- `decision_manager → alternative_agent` (जब प्रयोगकर्ताले "हो" भन्छ)
- `decision_manager → cancellation_agent` (जब प्रयोगकर्ताले "होइन" भन्छ)
- `availability_agent → booking_agent` (कोठाहरू उपलब्ध हुँदा)
- सबै पथहरू `display_result` मा समाप्त हुन्छन्


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## चरण १०: टेस्ट केस १ चलाउनुहोस् - शहर बिना उपलब्धता (पेरिस मानव पुष्टिकरणसहित)

यो परीक्षणले **पूर्ण मानव-इन-द-लूप चक्र** देखाउँछ:

१। पेरिसमा होटल अनुरोध गर्नुहोस्
२। availability_agent जाँच गर्छ → कोठा छैन
३। confirmation_agent मानव-सामूने प्रश्न बनाउँछ
४। request_info_executor **कार्यप्रवाह रोक्छ** र `RequestInfoEvent` उत्सर्जन गर्छ
५। **एप्लिकेशनले घटना पत्ता लगाउँछ र कन्सोलमा प्रयोगकर्तालाई सोध्छ**
६। प्रयोगकर्ताले "हो" वा "होइन" टाइप गर्छ
७। एप्लिकेशनले प्रतिक्रिया `send_responses_streaming()` मार्फत पठाउँछ
८। decision_manager प्रतिक्रियाको आधारमा मार्गनिर्देशन गर्छ
९। अन्तिम परिणाम देखाइन्छ

**मुख्य ढाँचा:**
- पहिलो पुनरावृत्तिको लागि `workflow.run_stream()` प्रयोग गर्नुहोस्
- पछि पुनरावृत्तिहरूको लागि `workflow.send_responses_streaming(pending_responses)` प्रयोग गर्नुहोस्
- मानव इनपुट आवश्यक भएको बेला पत्ता लगाउन `RequestInfoEvent` सुन्नुहोस्
- अन्तिम परिणाम समात्न `WorkflowOutputEvent` सुन्नुहोस्


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## चरण ११: परीक्षण केस २ चलाउनुहोस् - शहर उपलब्धताको साथ (स्टकहोम - मानव इनपुट आवश्यक छैन)

यो परीक्षणले **प्रत्यक्ष बाटो** देखाउँछ जब कोठाहरु उपलब्ध छन्:

१. स्टकहोममा होटल अनुरोध गर्नुहोस्
२. availability_agent जाँच गर्छ → कोठाहरू उपलब्ध ✅
३. booking_agent बुकिङको सिफारिश गर्छ
४. display_result पुष्टिकरण देखाउँछ
५. **मानव इनपुट आवश्यक छैन!**

जब कोठाहरू उपलब्ध हुन्छन्, कार्यप्रवाहले पूर्ण रूपमा मानव-इन-द-लूप मार्गलाई छोड्छ।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## मुख्य कुरा र मानव-इन-द-लूप उत्तम अभ्यासहरू

### ✅ तपाईंले के सिक्नुभयो:

#### 1. **RequestInfoExecutor ढाँचा**
माइक्रोसफ्ट एजेन्ट फ्रेमवर्कमा मानव-इन-द-लूप ढाँचाले तीनवटा मुख्य कम्पोनेन्टहरू प्रयोग गर्दछ:
- `RequestInfoExecutor` - वर्कफ्लोलाई रोक्छ र इभेन्टहरू उत्सर्जन गर्दछ
- `RequestInfoMessage` - टाइप गरिएका अनुरोध पेलोडहरूको आधार वर्ग (यसको उप-वर्ग बनाउनुहोस्!)
- `RequestResponse` - मानवीय प्रतिक्रियाहरूलाई मूल अनुरोधहरूसँग सम्बन्धित बनाउँछ

**महत्त्वपूर्ण बुझाइ:**
- `RequestInfoExecutor` आफैंले इनपुट सङ्कलन गर्दैन - यो केवल वर्कफ्लो रोक्छ
- तपाईंको एप्लिकेसन कोडले `RequestInfoEvent` सुन्नुपर्छ र इनपुट सङ्कलन गर्नुपर्छ
- तपाईंले `send_responses_streaming()` लाई `request_id` लाई प्रयोगकर्ताको उत्तरसँग मिल्ने डिक्ट पठाउनुपर्छ

#### 2. **स्ट्रिमिङ कार्यान्वयन ढाँचा**
```python
# पहिलो पुनरावृत्ति
stream = workflow.run_stream(initial_request)

# त्यसपछिका पुनरावृत्तिहरू (मानव इनपुट पछि)
stream = workflow.send_responses_streaming(pending_responses)

# सधैं घटनाहरू प्रक्रिया गर्नुहोस्
events = [event async for event in stream]
```

#### 3. **इभेन्ट-चालित वास्तुकला**
वर्कफ्लोलाई नियन्त्रण गर्न विशिष्ट इभेन्टहरू सुन्नुहोस्:
- `RequestInfoEvent` - मानवीय इनपुट आवश्यक (वर्कफ्लो रोकियो)
- `WorkflowOutputEvent` - अन्तिम नतिजा उपलब्ध (वर्कफ्लो पूरा)
- `WorkflowStatusEvent` - अवस्था परिवर्तनहरू (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, आदि)

#### 4. **@handler सहित कस्टम निष्पादकहरू**
`DecisionManager` ले देखाउँछ कि कसरी निष्पादकहरू बनाउने:
- `@handler` डेकोरेटर प्रयोग गरेर धेरै कार्यहरू वर्कफ्लो चरणहरूका रूपमा उपलब्ध गराउने
- टाइप गरिएका सन्देशहरू प्राप्त गर्ने (जस्तै, `RequestResponse[HumanFeedbackRequest, str]`)
- सन्देशहरू अन्य निष्पादकहरूलाई पठाएर वर्कफ्लो मार्गनिर्देशन गर्ने
- `WorkflowContext` मार्फत सन्दर्भमा पहुँच गर्ने

#### 5. **मानव निर्णयहरूसँग सशर्त मार्गनिर्देशन**
तपाईं मानवीय प्रतिक्रियाहरू मूल्याङ्कन गर्ने सर्तहरू सिर्जना गर्न सक्नुहुन्छ:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 व्यावहारिक प्रयोगहरू:

1. **अनुमोदन वर्कफ्लोहरू**
   - खर्च रिपोर्टहरू प्रक्रिया गर्नु अघि व्यवस्थापक अनुमोदन लिनुहोस्
   - स्वत: इमेलहरू पठाउनु अघि मानवीय समीक्षा आवश्यक
   - उच्च-मूल्य लेनदेनहरू कार्यान्वयन गर्नु अघि पुष्टि गर्नुहोस्

2. **सामग्री मडरेशन**
   - संदेहजनक सामग्रीलाई मानवीय समीक्षा लागि झण्डा लगाउनुहोस्
   - कडा मामिलाहरूमा अन्तिम निर्णय लिन मडरेटरहरूलाई सोध्नुहोस्
   - AI को विश्वसनीयता कम हुँदा मानवीयमा मुद्दा अगाडि बढाउनुहोस्

3. **ग्राहक सेवा**
   - नियमित प्रश्नहरूलाई AI स्वचालित रूपमा व्यवस्थापन गर्न दिनुहोस्
   - जटिल मुद्दाहरूलाई मानवीय एजेन्टहरूमा अगाडि बढाउनुहोस्
   - ग्राहकलाई सोध्नुहोस् कि उनीहरू मानवसँग कुरा गर्न चाहन्छन् कि छैनन्

4. **डाटा प्रक्रिया**
   - अस्पष्ट डाटा प्रविष्टिहरू समाधान गर्न मानवीयहरूलाई सोध्नुहोस्
   - अस्पष्ट दस्तावेजहरूको AI व्याख्या पुष्टि गर्नुहोस्
   - प्रयोगकर्ताहरूलाई विभिन्न वैध व्याख्याहरू बीच छनोट गर्न दिनुहोस्

5. **सुरक्षा-महत्त्वपूर्ण प्रणालीहरू**
   - अविवर्तनीय कार्यहरू अघि मानवीय पुष्टि आवश्यक
   - संवेदनशील डाटामा पहुँच गर्नु अघि अनुमोदन लिनुहोस्
   - नियमन गरिएका उद्योगहरूमा निर्णयहरू पुष्टि गर्नुहोस् (स्वास्थ्य सेवा, वित्त)

6. **इन्टरएक्टिव एजेन्टहरू**
   - अर्को प्रश्नहरू सोध्ने संवादात्मक बोटहरू निर्माण गर्नुहोस्
   - जटिल प्रक्रियाहरूमा प्रयोगकर्ताहरूलाई मार्गदर्शन गर्ने विजार्डहरू सिर्जना गर्नुहोस्
   - मानवीयसँग चरण-दर-चरण सहकार्य गर्ने एजेन्टहरू डिजाइन गर्नुहोस्

### 🔄 तुलना: मानव-इन-द-लूपसहित र बिना

| सुविधा | सशर्त वर्कफ्लो | मानव-इन-द-लूप वर्कफ्लो |
|---------|---------------------|---------------------------|
| **कार्यन्वयन** | एकल `workflow.run()` | `run_stream()` + `send_responses_streaming()` सहित लूप |
| **प्रयोगकर्ता इनपुट** | छैन (पूर्ण रूपमा स्वचालित) | `input()` वा UI द्वारा अन्तरक्रियात्मक प्रम्प्टहरू |
| **कम्पोनेन्टहरू** | एजेन्टहरू + निष्पादकहरू | + RequestInfoExecutor + DecisionManager |
| **इभेन्टहरू** | AgentExecutorResponse मात्र | RequestInfoEvent, WorkflowOutputEvent, आदि |
| **रोकावट** | रोकावट छैन | RequestInfoExecutor मा वर्कफ्लो रोकिन्छ |
| **मानव नियन्त्रण** | मानव नियन्त्रण छैन | मान्छेले मुख्य निर्णयहरू गर्छन् |
| **प्रयोग मामला** | स्वचालन | सहकार्य र निरीक्षण |

### 🚀 उन्नत ढाँचाहरू:

#### धेरै मानवीय निर्णय बिन्दुहरू
तपाईं एउटै वर्कफ्लोमा धेरै `RequestInfoExecutor` नोडहरू राख्न सक्नुहुन्छ:
```python
.add_edge(agent1, request_info_1)  # पहिलो मानवीय निर्णय
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # दोस्रो मानवीय निर्णय
.add_edge(decision_manager_2, final_agent)
```

#### टाइमआउट ह्यान्डलिङ
मानवीय प्रतिक्रियाहरूको लागि टाइमआउटहरू लागू गर्नुहोस्:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # सुरक्षित विकल्पमा पूर्वनिर्धारित
```

#### सम्पन्न UI एकीकरण
`input()` को सट्टा वेब UI, Slack, Teams, आदि सँग एकीकृत गर्नुहोस्:
```python
if isinstance(event, RequestInfoEvent):
    # प्रयोगकर्ताको मनपर्ने च्यानलमा सूचना पठाउनुहोस्
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### सशर्त मानव-इन-द-लूप
विशेष परिस्थितिहरूमा मात्र मानवीय इनपुट माग्नुहोस्:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # मात्र त्यहां मानवलाई पठाउनुहोस् यदि विश्वास कम छ वा मान उच्च छ
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ उत्तम अभ्यासहरू:

1. **सधैं RequestInfoMessage को उपवर्ग बनाउनुहोस्**
   - प्रकार सुरक्षा र प्रमाणीकरण प्रदान गर्दछ
   - UI प्रदर्शनको लागि समृद्ध सन्दर्भ सक्षम पार्दछ
   - प्रत्येक अनुरोध प्रकारको उद्देश्य स्पष्ट पार्दछ

2. **वर्णनात्मक प्रम्प्टहरू प्रयोग गर्नुहोस्**
   - के सोध्दै हुनुहुन्छ भन्ने सन्दर्भ समावेश गर्नुहोस्
   - हरेक विकल्पका परिणामहरू व्याख्या गर्नुहोस्
   - प्रश्नहरू सरल र स्पष्ट राख्नुहोस्

3. **अप्रत्याशित इनपुटलाई व्यवस्थापन गर्नुहोस्**
   - प्रयोगकर्ताका प्रतिक्रियाहरू प्रमाणित गर्नुहोस्
   - अमान्य इनपुटका लागि पूर्वनिर्धारित मानहरू प्रदान गर्नुहोस्
   - स्पष्ट त्रुटि सन्देशहरू दिनुहोस्

4. **अनुरोध IDs ट्र्याक गर्नुहोस्**
   - request_id र प्रतिक्रियाबीचको सम्बन्ध प्रयोग गर्नुहोस्
   - अवस्थालाई म्यानुअली व्यवस्थापन गर्ने प्रयास नगर्नुहोस्

5. **ब्लक नगर्ने डिजाइन गर्नुहोस्**
   - इनपुटको प्रतिक्षामा थ्रेडहरू ब्लक नगर्नुहोस्
   - सम्पूर्ण रूपमा असिंक ढाँचाहरू प्रयोग गर्नुहोस्
   - साझा वर्कफ्लो इन्स्टेन्सहरूलाई समर्थन गर्नुहोस्

### 📚 सम्बन्धित अवधारणाहरू:

- **Agent Middleware** - एजेन्ट कलहरू अवरोध गर्ने (अघिल्लो नोटबुक)
- **Workflow State Management** - रनहरूको बीचमा वर्कफ्लो स्थिति कायम राख्ने
- **Multi-Agent Collaboration** - मानव-इन-द-लूपलाई एजेन्ट टोलीहरूसँग संयोजन गर्ने
- **इभेन्ट-चालित वास्तुकला** - इभेन्टहरूसँग प्रतिक्रियाशील प्रणालीहरू निर्माण गर्ने

---

### 🎓 बधाई छ!

तपाईंले माइक्रोसफ्ट एजेन्ट फ्रेमवर्कसँग मानव-इन-द-लूप वर्कफ्लोहरूमा महारत हासिल गर्नुभयो! अब तपाईं जान्नुहुन्छ कसरी:
- ✅ मानवीय इनपुट सङ्कलन गर्न वर्कफ्लोहरू रोक्ने
- ✅ RequestInfoExecutor र RequestInfoMessage प्रयोग गर्ने
- ✅ इभेन्टहरूसँग स्ट्रिमिङ कार्यान्वयन व्यवस्थापन गर्ने
- ✅ @handler सहित कस्टम निष्पादकहरू बनाउने
- ✅ मानवीय निर्णयहरूको आधारमा वर्कफ्लो मार्गनिर्देशन गर्ने
- ✅ मान्छेसँग सहकार्य गर्ने अन्तरक्रियात्मक AI एजेन्टहरू निर्माण गर्ने

**यो विश्वसनीय, नियन्त्रणयोग्य AI प्रणालीहरू निर्माण गर्नको लागि एउटा अत्यन्त महत्त्वपूर्ण ढाँचा हो!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
